# Tutorial 03: Make Training Fast with Async

Each Tinker API call involves a network round-trip plus GPU compute on a remote worker pool. With synchronous code, your process sits idle while Tinker works. With async, you can overlap requests so Tinker's workers stay busy.

Tinker processes requests in discrete **clock cycles** (~10 seconds each). If you don't have a request queued when a cycle starts, you miss that cycle entirely. This tutorial shows how to avoid that.

In [ ]:
import time

import numpy as np
import tinker

from tinker_cookbook.renderers import get_renderer
from tinker_cookbook.supervised.data import conversation_to_datum

## Setup

Same setup as Tutorial 02 -- create a training client and prepare a small batch of data.

In [ ]:
base_model = "Qwen/Qwen3-30B-A3B"

service_client = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(base_model=base_model, rank=16)
tokenizer = training_client.get_tokenizer()
renderer = get_renderer("qwen3", tokenizer)

# A small batch of training data
conversations = [
    [
        {"role": "user", "content": "Translate to French: good morning"},
        {"role": "assistant", "content": "bonjour"},
    ],
    [
        {"role": "user", "content": "Translate to French: thank you"},
        {"role": "assistant", "content": "merci"},
    ],
    [
        {"role": "user", "content": "Translate to French: goodbye"},
        {"role": "assistant", "content": "au revoir"},
    ],
]

batch = [conversation_to_datum(conv, renderer, max_length=256) for conv in conversations]
adam_params = tinker.AdamParams(learning_rate=1e-4)

## The sync baseline

In sync mode, each call blocks until completion. The `forward_backward` call returns an `APIFuture`; calling `.result()` blocks until the GPU work finishes. Only then do we submit `optim_step`.

In [ ]:
n_steps = 4

start = time.time()
for step in range(n_steps):
    # Submit forward_backward, then block until it completes
    fwd_bwd_future = training_client.forward_backward(batch, "cross_entropy")
    fwd_bwd_result = fwd_bwd_future.result()

    # Only now submit optim_step, then block again
    optim_future = training_client.optim_step(adam_params)
    optim_result = optim_future.result()

    loss = -np.mean([o["logprobs"].to_numpy().mean() for o in fwd_bwd_result.loss_fn_outputs])
    print(f"Step {step}: loss={loss:.4f}")

sync_time = time.time() - start
print(f"\nSync total: {sync_time:.1f}s ({sync_time / n_steps:.1f}s per step)")

## The double-await pattern

Tinker's async methods use a **double-await** pattern:

1. **First await** -- submits the request to Tinker and returns an `APIFuture`. The request is now queued.
2. **Second await** -- waits for the result to come back.

The key insight: after the first await, you can immediately submit more requests before awaiting any results. This lets Tinker process `forward_backward` and `optim_step` in the **same clock cycle** instead of separate ones.

```
Naive (3 clock cycles):    |--fwd_bwd--|  (wait)  |--optim--|  (wait)  |--next fwd_bwd--|
Double-await (1 cycle):    |--fwd_bwd + optim--|  |--next fwd_bwd + optim--|
```

In [ ]:
start = time.time()
for step in range(n_steps):
    # Submit both requests back-to-back (first awaits only)
    fwd_bwd_future = await training_client.forward_backward_async(batch, "cross_entropy")
    optim_future = await training_client.optim_step_async(adam_params)

    # Now await results -- both were processed in the same clock cycle
    fwd_bwd_result = await fwd_bwd_future
    optim_result = await optim_future

    loss = -np.mean([o["logprobs"].to_numpy().mean() for o in fwd_bwd_result.loss_fn_outputs])
    print(f"Step {step}: loss={loss:.4f}")

async_time = time.time() - start
print(f"\nAsync total: {async_time:.1f}s ({async_time / n_steps:.1f}s per step)")
print(f"Speedup vs sync: {sync_time / async_time:.1f}x")

## Pipelining across steps

We can go further: submit step N+1's `forward_backward` before awaiting step N's results. This ensures there is always a request queued when the next clock cycle starts, so we never waste a cycle.

```
Without pipelining:   |--step 0--|  (gap)  |--step 1--|  (gap)  |--step 2--|
With pipelining:      |--step 0--|--step 1--|--step 2--|--step 3--|
```

In [ ]:
start = time.time()

# Submit the first step's requests
fwd_bwd_future = await training_client.forward_backward_async(batch, "cross_entropy")
optim_future = await training_client.optim_step_async(adam_params)

for step in range(n_steps):
    is_last = step == n_steps - 1

    if not is_last:
        # Submit the NEXT step's forward_backward before awaiting current results.
        # This keeps the pipeline full -- Tinker always has work queued.
        next_fwd_bwd_future = await training_client.forward_backward_async(batch, "cross_entropy")
        next_optim_future = await training_client.optim_step_async(adam_params)

    # Now await the current step's results
    fwd_bwd_result = await fwd_bwd_future
    optim_result = await optim_future

    loss = -np.mean([o["logprobs"].to_numpy().mean() for o in fwd_bwd_result.loss_fn_outputs])
    print(f"Step {step}: loss={loss:.4f}")

    if not is_last:
        fwd_bwd_future = next_fwd_bwd_future
        optim_future = next_optim_future

pipelined_time = time.time() - start
print(f"\nPipelined total: {pipelined_time:.1f}s ({pipelined_time / n_steps:.1f}s per step)")
print(f"Speedup vs sync: {sync_time / pipelined_time:.1f}x")

## When to use async

**Always for training loops.** The overhead is minimal and the speedup can be 2-3x. The built-in recipes in `tinker_cookbook` all use async internally.

The double-await pattern becomes even more important in RL training, where you overlap sampling, environment steps, and training updates across multiple batches.

## Next steps

- **Tutorial 04** (`04_first_rl.ipynb`): Reinforcement learning, where async pipelining is essential.
- **Async docs** (`docs/async.mdx`): Full reference for sync/async APIs and futures.
- **Under the hood** (`docs/under-the-hood.mdx`): Details on clock cycles and worker pools.